# Phase 1 — Data Preprocessing

## 1. Imports

In [27]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
import nltk
import re
import os

nltk.download('stopwords', quiet=True)

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 20)

## 2. Load Data
Only the **accepted** loans file is used — the rejected file has a different schema with no outcome label.

In [28]:
ACCEPTED_PATH = 'archive/accepted_2007_to_2018Q4.csv'

# Low-memory load to avoid RAM spike on the full ~2M row file
df = pd.read_csv(ACCEPTED_PATH, low_memory=False)

print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

Loaded: 2,260,701 rows × 151 columns


## 3. Build the Label Column
- **1 (positive / non-default):** Fully Paid, Current  
- **0 (negative / default):** Default, Charged Off, Late (any), In Grace Period  
- Drop rows where `loan_status` is NaN or ambiguous (e.g. "Does not meet the credit policy")

In [29]:
print('loan_status value counts:')
print(df['loan_status'].value_counts())

loan_status value counts:
loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64


In [30]:
POSITIVE_STATUSES = {'Fully Paid', 'Current'}
NEGATIVE_STATUSES = {
    'Default',
    'Charged Off',
    'Late (31-120 days)',
    'Late (16-30 days)',
    'In Grace Period'
}

keep_mask = df['loan_status'].isin(POSITIVE_STATUSES | NEGATIVE_STATUSES)
df = df[keep_mask].copy()

df['label'] = df['loan_status'].apply(lambda x: 1 if x in POSITIVE_STATUSES else 0)

print(f'Rows after label filter: {len(df):,}')
print(f"Label distribution:\n{df['label'].value_counts()}")
print(f"Positive rate: {df['label'].mean():.1%}")

Rows after label filter: 2,257,919
Label distribution:
label
1    1955068
0     302851
Name: count, dtype: int64
Positive rate: 86.6%


## 4. Drop Post-Origination / Leakage Columns
These columns are only populated *after* a loan outcome is known and would cause data leakage.

In [31]:
LEAKAGE_COLS = [
    # Outcome / payment history
    'loan_status', 'out_prncp', 'out_prncp_inv',
    'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp',
    'total_rec_int', 'total_rec_late_fee', 'recoveries',
    'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt',
    'next_pymnt_d', 'last_credit_pull_d',
    # Post-hoc status flags
    'debt_settlement_flag', 'debt_settlement_flag_date',
    'settlement_status', 'settlement_date', 'settlement_amount',
    'settlement_percentage', 'settlement_term',
    'hardship_flag', 'hardship_type', 'hardship_reason',
    'hardship_status', 'hardship_start_date', 'hardship_end_date',
    'hardship_amount', 'hardship_length', 'hardship_dpd',
    'hardship_loan_status', 'hardship_payoff_balance_amount',
    'hardship_last_payment_amount', 'payment_plan_start_date',
    'orig_projected_additional_accrued_interest', 'deferral_term',
    # Last FICO (post-issue; paper explicitly avoids last_fico_range_low)
    'last_fico_range_high', 'last_fico_range_low',
]

df.drop(columns=LEAKAGE_COLS, inplace=True, errors='ignore')
print(f'Columns remaining: {df.shape[1]}')

Columns remaining: 114


## 5. Drop Administrative / ID Columns

In [32]:
ADMIN_COLS = [
    'id', 'member_id', 'url',
    'policy_code',       # always 1
    'pymnt_plan',        # almost always 'n'
    'disbursement_method',
    'funded_amnt_inv',   # near-duplicate of funded_amnt
]

df.drop(columns=ADMIN_COLS, inplace=True, errors='ignore')
print(f'Columns remaining: {df.shape[1]}')

Columns remaining: 107


## 6. Drop Columns with >10% Missing Values
As specified in the paper.

In [33]:
threshold = 0.90  # keep columns where >90% of values are present
before = df.shape[1]
df.dropna(axis=1, thresh=int(threshold * len(df)), inplace=True)
print(f'Dropped {before - df.shape[1]} columns with >10% missing. Remaining: {df.shape[1]}')

Dropped 37 columns with >10% missing. Remaining: 70


## 7. Parse & Clean Individual Columns

In [34]:
# --- term: ' 36 months' becomes 36 ---
df['term'] = df['term'].astype(str).str.extract(r'(\d+)').astype(float)

# --- int_rate: '13.99%' becomes 13.99 (may already be float) ---
if df['int_rate'].dtype == object:
    df['int_rate'] = df['int_rate'].astype(str).str.replace('%', '', regex=False).astype(float)

# --- revol_util: '29.7%' becomes 29.7 ---
if 'revol_util' in df.columns and df['revol_util'].dtype == object:
    df['revol_util'] = df['revol_util'].astype(str).str.replace('%', '', regex=False).astype(float)

# --- emp_length becomes ordinal integer ---
EMP_MAP = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3,
    '4 years': 4, '5 years': 5, '6 years': 6, '7 years': 7,
    '8 years': 8, '9 years': 9, '10+ years': 10
}
if 'emp_length' in df.columns:
    df['emp_length'] = df['emp_length'].map(EMP_MAP)

# --- Date columns becomes months since Jan 2007 (loan programme start) ---
REFERENCE_DATE = pd.Timestamp('2007-01-01')

def months_since(series, ref=REFERENCE_DATE):
    parsed = pd.to_datetime(series, format='%b-%Y', errors='coerce')
    return ((parsed - ref) / np.timedelta64(1, 'm')).round().astype(float)

for col in ['issue_d', 'earliest_cr_line']:
    if col in df.columns:
        df[col] = months_since(df[col])

print('Column parsing done.')
print(df[['term', 'int_rate', 'emp_length', 'issue_d', 'earliest_cr_line']].head(3))

Column parsing done.
   term  int_rate  emp_length    issue_d  earliest_cr_line
0  36.0     13.99        10.0  4688640.0        -1798560.0
1  36.0     11.99        10.0  4688640.0        -3726720.0
2  60.0     10.78        10.0  4688640.0        -3375360.0


## 8. TF-IDF on Loan Descriptions
Paper approach: stem, remove stopwords/punctuation/HTML, find top 20 words with highest absolute difference in TF-IDF score between defaulting and non-defaulting groups, then create binary presence features.

In [35]:
TFIDF_TOP_N = 20  # number of discriminative words to keep

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)       # strip HTML
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)   # remove punctuation/numbers
    return text.lower().strip()

stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def stem_tokenize(text):
    tokens = text.split()
    return ' '.join(
        stemmer.stem(t) for t in tokens if t not in stop_words and len(t) > 2
    )

tfidf_features = []

if 'desc' in df.columns:
    print('Running TF-IDF on loan descriptions...')

    df['desc_clean'] = df['desc'].apply(clean_text).apply(stem_tokenize)

    default_docs    = df.loc[df['label'] == 0, 'desc_clean'].tolist()
    nondefault_docs = df.loc[df['label'] == 1, 'desc_clean'].tolist()

    vectorizer = TfidfVectorizer(max_features=5000)
    vectorizer.fit(df['desc_clean'].tolist())
    vocab = vectorizer.get_feature_names_out()

    def group_tfidf_sum(docs):
        mat = vectorizer.transform(docs)
        return np.asarray(mat.sum(axis=0)).flatten()

    def normalize(arr):
        total = arr.sum()
        return arr / total if total > 0 else arr

    default_scores    = normalize(group_tfidf_sum(default_docs))
    nondefault_scores = normalize(group_tfidf_sum(nondefault_docs))

    diff = np.abs(default_scores - nondefault_scores)
    top_indices = np.argsort(diff)[-TFIDF_TOP_N:][::-1]
    top_words = vocab[top_indices]

    print(f'Top {TFIDF_TOP_N} discriminative words: {list(top_words)}')

    # Binary presence features
    for word in top_words:
        col_name = f'desc_has_{word}'
        df[col_name] = df['desc_clean'].str.contains(r'\b' + word + r'\b', regex=True).astype(int)
        tfidf_features.append(col_name)

    df.drop(columns=['desc', 'desc_clean'], inplace=True)
    print(f'Added {len(tfidf_features)} TF-IDF binary features.')
else:
    print('No desc column found — skipping TF-IDF step.')

No desc column found — skipping TF-IDF step.


## 9. Drop High-Cardinality Text Columns
These can't be directly one-hot encoded without memory explosion.

In [36]:
HIGH_CARD_COLS = [
    'emp_title',   # ~300k unique values
    'title',       # borrower-entered loan title, near-duplicate of purpose
    'zip_code',    # handled via census join below; too many values for OHE
]

df.drop(columns=HIGH_CARD_COLS, inplace=True, errors='ignore')
print(f'Columns remaining: {df.shape[1]}')

Columns remaining: 67


## 10. Feature Ablation — Drop Features That Hurt Specificity
From the paper's ablative analysis on the logistic model.

In [37]:
ABLATION_DROPS = [
    'last_fico_range_low',  # many zeros; explicitly called out in paper
    'installment',          # hurts specificity
    'open_acc',             # hurts specificity
    'int_rate',             # sporadic Lending Club adjustments = noisy
]

df.drop(columns=ABLATION_DROPS, inplace=True, errors='ignore')
print(f'Columns remaining after ablation: {df.shape[1]}')

Columns remaining after ablation: 64


## 11. Drop Remaining Rows with Missing Values
Paper reports ~3% of loans are dropped at this stage.

In [38]:
# How many rows would survive dropna?
print(f'Rows before dropna: {len(df):,}')
print(f'Rows after dropna:  {df.dropna().shape[0]:,}')
print()

# Which columns have ANY nulls, and how many?
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0].sort_values(ascending=False)
print(f'Columns still containing nulls ({len(null_counts)} total):')
print(null_counts.to_string())

Rows before dropna: 2,257,919
Rows after dropna:  1,888,699

Columns still containing nulls (41 total):
num_tl_120dpd_2m              150908
emp_length                    146873
mo_sin_old_il_acct            136322
bc_util                        73322
percent_bc_gt_75               72630
bc_open_to_buy                 72186
mths_since_recent_bc           70663
pct_tl_nvr_dlq                 67682
avg_cur_bal                    67597
mo_sin_old_rev_tl_op           67528
mo_sin_rcnt_rev_tl_op          67528
num_rev_accts                  67528
num_tl_90g_dpd_24m             67527
num_tl_30dpd                   67527
num_tl_op_past_12m             67527
num_actv_rev_tl                67527
num_rev_tl_bal_gt_0            67527
tot_hi_cred_lim                67527
num_op_rev_tl                  67527
num_il_tl                      67527
num_bc_tl                      67527
num_actv_bc_tl                 67527
tot_cur_bal                    67527
mo_sin_rcnt_tl                 67527
tot_coll

In [39]:
before = len(df)
df.dropna(inplace=True)
dropped_pct = (before - len(df)) / before
print(f'Dropped {before - len(df):,} rows ({dropped_pct:.1%}). Remaining: {len(df):,}')

Dropped 369,220 rows (16.4%). Remaining: 1,888,699


## 12. One-Hot Encode Low-Cardinality Categoricals
Only encode columns with manageable unique counts. This is safe now that high-cardinality columns are gone.

In [40]:
# Identify remaining object columns
obj_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Object columns to encode: {obj_cols}')

# Safety check: print cardinality before encoding
for c in obj_cols:
    print(f'  {c}: {df[c].nunique()} unique values')

Object columns to encode: ['grade', 'sub_grade', 'home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type']
  grade: 7 unique values
  sub_grade: 35 unique values
  home_ownership: 6 unique values
  verification_status: 3 unique values
  purpose: 14 unique values
  addr_state: 51 unique values


/var/folders/gv/51stnj090154tr_l71zx7gjw0000gn/T/ipykernel_21550/3342537697.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(include='object').columns.tolist()


  initial_list_status: 2 unique values
  application_type: 2 unique values


In [41]:
# Encode columns with ≤55 unique values (covers grade, sub_grade, home_ownership,
# verification_status, purpose, addr_state, initial_list_status, application_type, term if str)
SAFE_CAT_COLS = [c for c in obj_cols if df[c].nunique() <= 55]
print(f'Encoding: {SAFE_CAT_COLS}')

df = pd.get_dummies(df, columns=SAFE_CAT_COLS, drop_first=True)
print(f'Shape after one-hot encoding: {df.shape}')

Encoding: ['grade', 'sub_grade', 'home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type']
Shape after one-hot encoding: (1888699, 168)


In [42]:
# Any remaining object columns that slipped through — inspect and drop
still_obj = df.select_dtypes(include='object').columns.tolist()
if still_obj:
    print(f'WARNING: still-object columns (dropping): {still_obj}')
    df.drop(columns=still_obj, inplace=True)
else:
    print('All object columns resolved.')

All object columns resolved.


## 13. Train / Test Split
70/30 time-ordered split — **no shuffle**, since loans are ordered chronologically.

In [43]:
split_idx = int(len(df) * 0.70)

train_df = df.iloc[:split_idx].copy()
test_df  = df.iloc[split_idx:].copy()

X_train = train_df.drop(columns=['label'])
y_train = train_df['label']

X_test  = test_df.drop(columns=['label'])
y_test  = test_df['label']

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train positive rate: {y_train.mean():.1%}')
print(f'Test  positive rate: {y_test.mean():.1%}')

Train: (1322089, 167)  |  Test: (566610, 167)
Train positive rate: 87.0%
Test  positive rate: 87.2%


## 14. Feature Scaling (for SVM / Logistic Regression)
Scale to [-1, 1] as described in Section VII of the paper. Fit only on training data.

In [44]:
from sklearn.preprocessing import MinMaxScaler

# One hot encoded columns are binary (0/1) — scaling them distorts their meaning, so only continuous numeric columns are scaled to [-1, 1].
ohe_cols  = [c for c in X_train.columns if X_train[c].nunique() == 2 and set(X_train[c].unique()).issubset({0, 1})]
cont_cols = [c for c in X_train.columns if c not in ohe_cols]

print(f'Continuous columns to scale: {len(cont_cols)}')
print(f'Binary OHE columns (pass-through): {len(ohe_cols)}')

scaler = MinMaxScaler(feature_range=(-1, 1))

# Scale only continuous columns, then recombine with binary columns
X_train_scaled = X_train.copy()
X_train_scaled[cont_cols] = scaler.fit_transform(X_train[cont_cols])

X_test_scaled = X_test.copy()
X_test_scaled[cont_cols] = scaler.transform(X_test[cont_cols])

print('Scaling done.')
print(f'Continuous range check — min: {X_train_scaled[cont_cols].min().min():.2f}, max: {X_train_scaled[cont_cols].max().max():.2f}')
print(f'OHE range check — min: {X_train_scaled[ohe_cols].min().min():.2f}, max: {X_train_scaled[ohe_cols].max().max():.2f}')

Continuous columns to scale: 56
Binary OHE columns (pass-through): 111
Scaling done.
Continuous range check — min: -1.00, max: 1.00
OHE range check — min: 0.00, max: 1.00


## 15. Save Preprocessed Data

In [45]:
OUTPUT_DIR = 'preprocessed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

X_train.to_csv(f'{OUTPUT_DIR}/X_train.csv', index=False)
X_test.to_csv(f'{OUTPUT_DIR}/X_test.csv',  index=False)
y_train.to_csv(f'{OUTPUT_DIR}/y_train.csv', index=False)
y_test.to_csv(f'{OUTPUT_DIR}/y_test.csv',  index=False)

X_train_scaled.to_csv(f'{OUTPUT_DIR}/X_train_scaled.csv', index=False)
X_test_scaled.to_csv(f'{OUTPUT_DIR}/X_test_scaled.csv',  index=False)

print('Saved to ./preprocessed/')
print('Files:')
for f in os.listdir(OUTPUT_DIR):
    size_mb = os.path.getsize(f'{OUTPUT_DIR}/{f}') / 1e6
    print(f'  {f}  ({size_mb:.1f} MB)')

Saved to ./preprocessed/
Files:
  X_train_scaled.csv  (1941.5 MB)
  X_train.csv  (1275.7 MB)
  y_train.csv  (2.6 MB)
  y_test.csv  (1.1 MB)
  X_test.csv  (546.6 MB)
  X_test_scaled.csv  (831.6 MB)


## 16. Summary

In [46]:
print('=== PREPROCESSING SUMMARY ===')
print(f'Total features:         {X_train.shape[1]}')
print(f'Training samples:       {len(X_train):,}')
print(f'Test samples:           {len(X_test):,}')
print(f'Train positive rate:    {y_train.mean():.1%}')
print(f'Test  positive rate:    {y_test.mean():.1%}')
print()
print('Outputs:')
print('  X_train / X_test          — unscaled (for Naive Bayes / tree models)')
print('  X_train_scaled / X_test_scaled — [-1,1] scaled (for SVM / Logistic Regression)')
print('  y_train / y_test          — binary labels (1=non-default, 0=default)')

=== PREPROCESSING SUMMARY ===
Total features:         167
Training samples:       1,322,089
Test samples:           566,610
Train positive rate:    87.0%
Test  positive rate:    87.2%

Outputs:
  X_train / X_test          — unscaled (for Naive Bayes / tree models)
  X_train_scaled / X_test_scaled — [-1,1] scaled (for SVM / Logistic Regression)
  y_train / y_test          — binary labels (1=non-default, 0=default)
